# Antioch: `humanoid_legs_5` setup and validation

This notebook is a **terminal runbook**. Open it in VS Code and copy each Bash block into a terminal on the Antioch machine. The linked robot is an inspection asset; finish the PhysX and hardware checks below before training a policy. Run commands in order.


## 1. Get the clean deployment branch

If the repo is not on the Antioch machine yet:

```bash
git clone --depth 1 --single-branch --branch antioch/humanoid_legs_5 https://github.com/triton-droids/simulation.git
cd simulation
```

If it is already cloned:

```bash
cd /path/to/simulation
git fetch origin
git switch --track origin/antioch/humanoid_legs_5
git pull --ff-only
```

For later updates, stay on this branch and run `git pull --ff-only`. Check the selected commit with `git status -sb` and `git log -1 --oneline`.


## 2. Confirm the GPU and files

```bash
nvidia-smi
ls -lh robots/humanoid_legs_5/
ls robots/humanoid_legs_5/meshes/*.stl | wc -l
```

Expect 24 meshes, `humanoid_legs_5.xml`, two MP4 sweeps, and the validation report. The videos are a kinematic sweep, not a learned walking rollout.


## 3. Recheck the portable MJCF

If Antioch gives you a Python environment with MuJoCo and NumPy, use it. Otherwise make a small local virtual environment:

```bash
python3 -m venv .venv-check
source .venv-check/bin/activate
python -m pip install --upgrade pip
python -m pip install mujoco numpy
python robots/humanoid_legs_5/validate_mjcf.py
deactivate
```

The expected result is 10 actuators, 21 `qpos` entries, approximately 16 kg, and a maximum driven pin gap below 0.1 mm. This only verifies the MJCF in MuJoCo.


## 4. Identify Antioch's Isaac installation

Run these in the Antioch terminal. Use the Isaac Lab environment provided there; do not install a second Isaac Lab blindly.

```bash
python -c "import isaaclab; print(isaaclab.__file__)"
python -c "import isaacsim; print(isaacsim.__file__)"
find / -path '*/scripts/tools/convert_mjcf.py' -print 2>/dev/null | head
```

Record the Isaac Sim and Isaac Lab versions shown in Antioch's environment. Their compatible pair matters. The commands below use the current Isaac Lab source checkout layout; adjust the launcher if Antioch supplies its own wrapper.


## 5. Convert MJCF to USD with Isaac Lab

Set paths in the **same terminal**. Replace `/path/to/IsaacLab` with the checkout found above. Use the PhysX-enabled Isaac Sim/Isaac Lab environment provided by Antioch.

```bash
export SIM_REPO="$(pwd)"
export ISAACLAB_ROOT=/path/to/IsaacLab
cd "$ISAACLAB_ROOT"
uv run --extra importers isaaclab -p scripts/tools/convert_mjcf.py \
  "$SIM_REPO/robots/humanoid_legs_5/humanoid_legs_5.xml" \
  "$SIM_REPO/outputs/humanoid_legs_5.usd" \
  --viz kit
```

If `uv` is unavailable, use Antioch's Isaac Lab Python launcher with `scripts/tools/convert_mjcf.py`; first run that script with `--help` to confirm its arguments. Keep generated USD under `outputs/` and out of Git. The conversion command follows the [Isaac Lab importer guide](https://isaac-sim.github.io/IsaacLab/main/source/how-to/import_new_asset.html).


## 6. Inspect the PhysX asset before any training

In the Isaac Sim viewport, check both bars through the neutral pose and ankle sweep. Inspect the USD stage and articulation: two heel bars, two cranks, both pin connections, ten driven joints, and four passive joints. Step under gravity with ground contact. Watch for pin separation, startup jumps, self-intersection, foot penetration, or unstable motion. Check reported mass and inertia and verify that the imported joint names and axes match the MJCF.

**Stop the migration if the importer drops or changes a loop constraint.** MuJoCo closure does not prove PhysX closure. Do not choose Newton for this closed-chain model; [NVIDIA currently lists closed loops as unsupported there](https://docs.isaacsim.omniverse.nvidia.com/6.2.0/physics/newton_physics.html).


## 7. Hardware and training gates

Before building the Isaac Lab locomotion task, measure the actual motor shaft axes, linkage zero offsets, joint travel, backlash/friction, motor-to-foot angle and torque mapping, and rotor inertia. Reconcile the foot/bar inertias from CAD or physical measurements. Confirm the left CAD upper-pin mismatch (2.649 mm) against the real assembly.

Then create an Isaac Lab articulation config and a velocity-command task. Select the **ten controlled joints by name**; never assume all 14 hinge joints are policy actions. Match the deployed observation order, 50 Hz control, actuator limits, and target slew to the hardware. First test standing and short commanded walking with a small environment count. Scale up on the RTX PRO 6000 only after the small PhysX run is stable. SuperMimic or Puppeteer can follow a reliable baseline and suitable motion data.


## Evidence in this branch

- [MJCF](../robots/humanoid_legs_5/humanoid_legs_5.xml)
- [Validation report](../robots/humanoid_legs_5/humanoid_legs_5_validation.json)
- [Full robot sweep](../robots/humanoid_legs_5/humanoid_legs_5_full_sweep.mp4)
- [Ankle detail sweep](../robots/humanoid_legs_5/humanoid_legs_5_ankle_sweep.mp4)
- [Model limits](../robots/humanoid_legs_5/README.md)
